# Modelado — predicción de supervivencia

Segundo cuaderno del taller. El primero (`02_exploracion.ipynb`) respondía
*"¿qué se asocia con sobrevivir?"*; este responde *"dado un pasajero nuevo,
¿qué probabilidad tiene de sobrevivir?"*.

Lee la capa **Silver** a través de `flows/05_ml_models.py`, nunca los datos
directamente. Para reproducirlo desde cero:

```
make ingest && make silver && make gold
```

y luego *Restart & Run All*.

In [ ]:
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Los modulos con nombre numerico se importan con importlib (seccion 3 de la guia)
ml = importlib.import_module("flows.05_ml_models")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

X, y = ml.load_dataset()          # lee Silver; X con columnas candidatas, y = Survived
print(X.shape, round(y.mean(), 4))

## Parte 10 — Formulación del problema

Antes de entrenar nada hay que fijar cuatro cosas: la **variable objetivo**, el
**momento de predicción**, la **métrica** y el **baseline** a superar.

El orden importa. Si la métrica se define *después* de entrenar, es fácil
elegir la que mejor le quede al modelo que ya se tiene.

El documento completo está en `reports/problema_ml.md`.

### Paso 39. Qué se predice y cómo se mide

In [ ]:
print(f"Pasajeros:             {len(X)}")
print(f"Variables candidatas:  {X.shape[1]}")
print(f"Tasa de supervivencia: {y.mean():.4f}")
print()
print("Variables que entran al modelo:")
print(f"  numericas:   {ml.NUM_FEATURES}")
print(f"  categoricas: {ml.CAT_FEATURES}")
print(f"  sospechosas: {ml.CABIN_NUM + ml.CABIN_CAT}   <- se evaluan en el Paso 61")
print()
print(f"Metrica principal: {ml.PRIMARY_METRIC}")

### Paso 40. Auditoría de variables y fuga de datos

| Columna | ¿Existe al embarcar? | ¿Riesgo de fuga? | Decisión |
|---|---|---|---|
| `Survived`, `Survivor` | No, es el objetivo | **Fuga directa** | Excluir |
| `PassengerId`, `Ticket`, `Name` | Sí | Identificadores: se memorizan, no generalizan | Excluir (el título entra vía `TTitulo`) |
| `NSex`, `NEmbarked`, `NCabin`, `NTitulo` | — | Duplican a sus categóricas e imponen un orden que no existe | Excluir; se usa *one-hot* |
| `Age` imputada | — | Calculada con el promedio de **todos** los pasajeros | Usar `AgeOriginal` |
| `Person` | Derivada de `Age` y `Sex` | Redundante con ambas | Excluir del modelo base |
| `Cabin`, `TCabin`, `HasCabin` | **Dudoso** | ¿Se registró la cabina *después* del naufragio? | Evaluar con y sin ella (Paso 61) |

**La fuga más sutil es la edad.** La `Age` de Silver se imputó con el promedio
de los 891 pasajeros, incluidos los que terminarán en el conjunto de prueba.
Entrenar con ella le daría al modelo información que no debería tener. Por eso
`load_dataset()` recupera `AgeOriginal`, con sus nulos, y la imputación se hace
dentro del Pipeline, que la calcula solo con datos de entrenamiento.

In [ ]:
# Verificacion: la columna Age que entra al modelo conserva los nulos reales
print(f"Nulos en la Age del modelo: {X['Age'].isna().sum()}   (deben ser 177)")
print()
silver = pd.read_parquet(ml.SILVER / ml.SILVER_NAME)
print("Columnas de Silver que NO entran al modelo:")
for c in sorted(set(silver.columns) - set(X.columns)):
    print(f"  {c}")

**La pregunta de debate del Paso 40.**

Si `HasCabin` mejora la métrica, ¿significa que tener cabina protegía?

Probablemente no. El Paso 5 mostró que la cabina falta en el 18.5 % de primera
clase y en el 97.6 % de tercera: la ausencia del dato refleja **cómo se
compilaron los manifiestos**, no una característica del pasajero. Y cabe la
posibilidad de que se registrara con más detalle para quienes sobrevivieron
para declararla.

Un modelo que explota un artefacto del registro puede tener buenas métricas y
ser inútil para estimar riesgo real. Por eso `TCabin` y `HasCabin` quedan
marcadas como sospechosas y se evalúan por separado en el Paso 61.

### Paso 41. División estratificada

El conjunto de prueba es una evaluación **final** y se abre **una sola vez**,
en la Parte 16. Todas las decisiones intermedias —qué modelo, qué
hiperparámetros, qué umbral, qué variables— se toman con validación cruzada
sobre el entrenamiento.

`stratify=y` mantiene la proporción de sobrevivientes en ambos conjuntos.
`random_state` fija la semilla para que la división sea reproducible.

In [ ]:
X_train, X_test, y_train, y_test = ml.split(X, y)

print(f"Entrenamiento: {len(X_train)} pasajeros, tasa {y_train.mean():.4f}")
print(f"Prueba:        {len(X_test)} pasajeros, tasa {y_test.mean():.4f}")
print(f"Completo:      {len(X)} pasajeros, tasa {y.mean():.4f}")
print()
print(f"Diferencia entre tasas: {abs(y_train.mean() - y_test.mean()):.4f}")

> ⚠️ **A partir de aquí, `X_test` y `y_test` no se tocan** hasta la Parte 16.
> Mirarlos antes invalidaría la evaluación final: cualquier decisión tomada
> viéndolos ajustaría el modelo a ese conjunto concreto, y sus métricas
> dejarían de estimar el rendimiento sobre datos nuevos.

### Paso 42. Baselines: la vara a superar

Un modelo complejo que no supera a una regla simple no aporta valor.

- **Baseline ingenuo:** predecir siempre "no sobrevive".
- **Regla de género:** predecir que sobrevive toda mujer. Es la regla que
  cualquiera deduciría del EDA en cinco minutos.

In [ ]:
ml.baseline_scores(X_train, y_train)

**Lo que muestran estos dos números.**

El baseline ingenuo acierta en torno al 62 % **sin aprender nada**, y su recall
es exactamente 0: no identifica ni un solo superviviente. Es la prueba de que
**la exactitud sola engaña** cuando las clases están desbalanceadas.

La regla de género ronda el 79 % de exactitud. **Esa es la vara real.** Un
modelo de machine learning que quede por debajo no justifica su complejidad.

Por eso la métrica principal es **ROC-AUC** y no la exactitud: no depende del
umbral de decisión y mide la capacidad de ordenar pasajeros por riesgo, que es
lo que una aseguradora necesita.

In [ ]:
# Para tenerlo a mano al comparar modelos en la Parte 13
baselines = ml.baseline_scores(X_train, y_train)
VARA = baselines.set_index("baseline").loc["regla de genero", "accuracy"]
print(f"Vara a superar (exactitud de la regla de genero): {VARA:.4f}")

## Parte 11 — Preprocesamiento reproducible con `Pipeline`

Un `Pipeline` encadena los pasos —imputar, codificar, modelar— en **un solo
objeto** con `fit` y `predict`. Al hacer `fit`, cada paso aprende solo de los
datos que recibe; al predecir, aplica lo aprendido.

Eso garantiza el mismo preprocesamiento en entrenamiento, validación y
producción, sin duplicar código y sin que una estadística del conjunto de
prueba se filtre al de entrenamiento.

### Paso 43. Demostrar la fuga de datos

El experimento: calcular el promedio de edad de las mujeres con **todos** los
datos y con **solo el entrenamiento**, y comparar.

In [ ]:
media_todos = X.loc[X["Sex"] == "female", "Age"].mean()
media_train = X_train.loc[X_train["Sex"] == "female", "Age"].mean()

print(f"Promedio de edad de mujeres, TODOS los datos: {media_todos:.4f}")
print(f"Promedio de edad de mujeres, solo TRAIN:      {media_train:.4f}")
print(f"Diferencia:                                   {abs(media_todos - media_train):.4f}")
print()
print(f"Mujeres en total: {(X['Sex'] == 'female').sum()}")
print(f"Mujeres en train: {(X_train['Sex'] == 'female').sum()}")
print(f"Mujeres en test:  {(X_test['Sex'] == 'female').sum()}")

**Por qué la diferencia es pequeña aquí, y por qué sigue siendo un error.**

La diferencia es de centésimas porque hay cientos de mujeres y el 80 % de
ellas ya está en el entrenamiento: añadir el 20 % restante casi no mueve el
promedio.

Pero el tamaño del error no es lo que lo hace incorrecto. Al usar el promedio
de todos, **información del conjunto de prueba viaja al entrenamiento**: el
modelo "sabe", indirectamente, algo sobre pasajeros que debería desconocer por
completo. La evaluación final deja de estimar el rendimiento sobre datos
nuevos y pasa a estimarlo sobre datos que el modelo ya vio de refilón.

Y en otros conjuntos de datos la diferencia **sí** es grande: con más
faltantes, con grupos pequeños, o con variables de alta varianza, una
estadística calculada sobre todo puede cambiar mucho el resultado. El método
debe ser correcto independientemente de que el error resulte pequeño en este
caso concreto.

### Paso 44. `GroupAgeImputer`

La imputación del Paso 22, ahora como transformador de scikit-learn. Esa
conversión es exactamente lo que elimina la fuga: `fit()` calcula los
promedios con los datos que recibe —dentro de la validación cruzada, solo los
de entrenamiento de ese pliegue— y `transform()` los aplica.

Si aparece un grupo que `fit` nunca vio, usa el promedio global en vez de
dejar un nulo.

In [ ]:
from sklearn.dummy import DummyClassifier

pipe = ml.build_pipeline(DummyClassifier())
pre = pipe[:-1].fit(X_train, y_train)      # solo el preprocesamiento, ajustado en TRAIN

print("Promedios de edad que aprendio el imputador, por (Sex, Pclass):")
print(pre["age_imputer"].group_means_.round(2).to_string())
print(f"\nPromedio global de respaldo: {pre['age_imputer'].global_mean_:.2f}")

### Paso 45-46. Verificar el preprocesamiento

El preprocesador se ajusta con `X_train` y se aplica a `X_test`. Tres
comprobaciones: que no queden nulos, cuántas columnas salen, y que las
categóricas no lleven un orden implícito.

In [ ]:
Z = pre.transform(X_test)

print(f"Entrada:  {X_test.shape[1]} columnas")
print(f"Salida:   {Z.shape[1]} columnas")
print(f"¿Quedan NaN tras el preprocesamiento? {np.isnan(Z).any()}")
print()
print("Nombres de las columnas generadas:")
for nombre in pre["prep"].get_feature_names_out():
    print(f"  {nombre}")

**Las dos cosas que confirma esta salida.**

**No quedan nulos.** Los 177 faltantes de edad los rellenó el `GroupAgeImputer`
con promedios aprendidos *solo* en entrenamiento. Si hubiera quedado algún NaN,
varios modelos fallarían al entrenar.

**`Sex` salió como `cat__Sex_female` y `cat__Sex_male`**, dos columnas de 0 y 1,
no como un único número. Eso es *one-hot*, y evita el problema que anotamos en
el Paso 24: una codificación numérica le diría al modelo que `Q` (2) es mayor
que `S` (0), un orden que no existe.

`min_frequency=10` agrupa las categorías muy raras. La cubierta `T`, con un
solo pasajero, no genera una columna propia que el modelo podría memorizar.

### La decisión a justificar: ¿agrupar por `Sex` o por `Sex` y `Pclass`?

El Paso 22 imputaba la edad con el promedio por género. El pipeline usa un
grupo más específico: género **y** clase.

La comparación formal con validación cruzada llega en el Paso 48. Por ahora,
cuánto difieren los valores que cada variante asignaría.

In [ ]:
por_sexo = ml.GroupAgeImputer(group_cols=("Sex",)).fit(X_train)
por_sexo_clase = ml.GroupAgeImputer(group_cols=("Sex", "Pclass")).fit(X_train)

print("Agrupando solo por Sex:")
print(por_sexo.group_means_.round(2).to_string())
print("\nAgrupando por Sex y Pclass:")
print(por_sexo_clase.group_means_.round(2).to_string())
print("\nRango de valores imputados:")
print(f"  solo Sex:        {por_sexo.group_means_.min():.2f} a {por_sexo.group_means_.max():.2f}")
print(f"  Sex y Pclass:    {por_sexo_clase.group_means_.min():.2f} a {por_sexo_clase.group_means_.max():.2f}")

La agrupación por `Sex` y `Pclass` produce un rango de valores más amplio,
porque recoge algo que el EDA ya mostraba: los pasajeros de primera clase eran
notablemente mayores que los de tercera (Paso 10). Imputar a una mujer de
tercera clase con el promedio de *todas* las mujeres le asignaría una edad
sesgada hacia arriba.

A cambio, los grupos son más pequeños y cada promedio se estima con menos
datos. El Paso 48 dirá si esa especificidad compensa.

## Parte 12 — Modelos candidatos y validación cruzada

Un modelo puede **memorizar** el entrenamiento y fallar con datos nuevos: eso
es sobreajuste. La validación cruzada de *k* pliegues divide el entrenamiento
en *k* partes, entrena con *k−1* y evalúa en la restante, rotando.

La **media** de los *k* puntajes estima el desempeño; la **desviación** indica
la estabilidad.

Con 712 pasajeros de entrenamiento la varianza es considerable. **Una
diferencia de medio punto entre dos modelos no es evidencia de que uno sea
mejor.**

### Paso 47. Los seis candidatos

In [ ]:
for nombre, modelo in ml.get_candidates().items():
    print(f"  {nombre:8s} {type(modelo).__name__}")

Una familia variada a propósito: un baseline que no aprende nada, un modelo
lineal interpretable, un árbol simple, dos ensambles y uno basado en
distancia. Todos con la semilla fija.

`dummy` está incluido deliberadamente: si un modelo no lo supera con holgura,
el problema está en los datos o en el preprocesamiento, no en el modelo.

### Paso 48. Validación cruzada

Cada candidato se evalúa dentro de **su propio pipeline completo**: la
imputación de edad, el escalado y el one-hot se recalculan en cada pliegue
usando solo los datos de entrenamiento de ese pliegue. Eso es lo que hace que
las métricas sean honestas.

In [ ]:
%%time
resultados, por_pliegue = ml.cross_validate_candidates(
    X_train, y_train, return_folds=True
)
resultados

### Paso 49. Leer los resultados

Tres lecturas, en este orden: desempeño, estabilidad y sobreajuste.

In [ ]:
cols = ["modelo", "roc_auc_mean", "roc_auc_std", "accuracy_mean", "f1_mean"]
tabla = resultados[cols].copy()
tabla["supera_la_vara"] = tabla["accuracy_mean"] > VARA
tabla.round(4)

In [ ]:
# Dispersión por pliegue de la métrica principal: la media sola esconde la variabilidad
auc = por_pliegue[por_pliegue["metrica"] == "roc_auc"]
orden = resultados.sort_values("roc_auc_mean", ascending=False)["modelo"]

sns.boxplot(data=auc, x="modelo", y="valor", order=orden)
sns.stripplot(data=auc, x="modelo", y="valor", order=orden, color="black", size=4)
plt.axhline(0.5, color="red", linestyle="--", linewidth=1, label="azar")
plt.title("ROC-AUC por pliegue")
plt.ylabel("ROC-AUC")
plt.xlabel("")
plt.legend()
plt.show()

In [ ]:
# Brecha entrenamiento - validacion: cuanto peor le va al modelo con datos que no vio
brecha = resultados[["modelo", "train_roc_auc_mean", "roc_auc_mean",
                     "brecha_roc_auc", "fit_time_mean"]].copy()
brecha = brecha.sort_values("brecha_roc_auc", ascending=False)
brecha.round(4)

**Cómo leer la brecha.**

Es el ROC-AUC en entrenamiento menos el de validación. Mide cuánto peor le va
al modelo con datos que no vio.

- **Brecha cercana a 0:** el modelo generaliza bien, o es demasiado simple
  para memorizar nada (el caso de `dummy`).
- **Brecha grande:** el modelo memorizó parte del entrenamiento. Suele crecer
  con la flexibilidad: un bosque de 300 árboles sin profundidad limitada puede
  ajustarse casi perfectamente a los datos que vio.

Un modelo con buena media y brecha pequeña es preferible a uno con media
ligeramente mejor y brecha grande: el segundo promete algo que no va a
cumplir con datos nuevos.

### La decisión pendiente del Paso 45: ¿`Sex` o `Sex` + `Pclass`?

Ahora sí se puede comparar con validación cruzada en lugar de con intuición.

In [ ]:
from sklearn.base import clone

def auc_con_imputador(group_cols):
    """Valida el mejor modelo lineal cambiando solo el agrupamiento del imputador."""
    pipe = ml.build_pipeline(ml.get_candidates()["logreg"])
    pipe.set_params(age_imputer__group_cols=group_cols)
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    s = cross_val_score(clone(pipe), X_train, y_train, cv=cv, scoring="roc_auc")
    return s.mean(), s.std()

for cols_grupo in [("Sex",), ("Sex", "Pclass")]:
    media, desv = auc_con_imputador(cols_grupo)
    print(f"  agrupando por {cols_grupo!s:20s} ROC-AUC = {media:.4f} (+/- {desv:.4f})")

### Paso 50. Elegir finalistas

El criterio **no** es "el primero de la tabla". Son cinco dimensiones:

1. **Desempeño medio** en la métrica principal.
2. **Estabilidad** — la desviación entre pliegues.
3. **Brecha entrenamiento-validación** — cuánto sobreajusta.
4. **Interpretabilidad** — ¿se puede explicar una predicción a la gerencia?
5. **Costo computacional** — `fit_time_mean`.

Un modelo ligeramente peor pero estable y explicable puede ser mejor elección
que el primero de la tabla.

> ⚠️ **Regla de oro: nunca se elige un modelo mirando el conjunto de prueba.**
> Todo lo anterior se decidió con validación cruzada sobre el entrenamiento.

La justificación escrita va en `reports/problema_ml.md`, sección "Selección de
modelos".

## Parte 13 — Evaluación en profundidad

Un solo número oculta **qué** errores comete el modelo. La Parte 12 dejó tres
finalistas y una tabla de medias; aquí se abre esa media para ver a quién
acierta y a quién no.

La herramienta es la **predicción fuera de muestra**: cada pasajero se predice
con un modelo entrenado sin él. Así se puede mirar matriz de confusión,
curvas, calibración y errores por subgrupo **sin tocar el conjunto de
prueba**, que sigue cerrado hasta la Parte 16.

> Una advertencia que conviene tener presente todo el tiempo: el vector de
> probabilidades mezcla los 5 modelos de los 5 pliegues. Diagnostica el
> **procedimiento**, no un modelo concreto.

### Paso 51. Predicciones fuera de muestra y matriz de confusión

In [ ]:
%%time
# Una probabilidad honesta por pasajero: ninguna la emitió un modelo que ya lo vio
proba_oof = ml.oof_proba_finalists(X_train, y_train)
proba_oof.describe().round(4)

In [ ]:
# Matriz de confusión del líder, con el umbral por convención
from sklearn.metrics import classification_report

LIDER = "hgb"
proba = proba_oof[LIDER].to_numpy()
pred = (proba >= 0.5).astype(int)

display(ml.confusion_table(y_train, pred))
print(classification_report(y_train, pred,
                            target_names=["no sobrevivió", "sobrevivió"], digits=3))

In [ ]:
# Qué significa cada celda para la aseguradora
vn, fp, fn, vp = ml.confusion_table(y_train, pred).to_numpy().ravel()
razon = f"{fp / fn:.2f}" if fn else "n/d"

print(f'''
Verdaderos negativos : {vn:3d}   no sobrevivió y el modelo lo previó
Falsos positivos     : {fp:3d}   EL ERROR CARO: póliza con prima insuficiente
Falsos negativos     : {fn:3d}   se cobra de más o se rechaza a un cliente viable
Verdaderos positivos : {vp:3d}   sobrevivió y el modelo lo previó

Por cada falso negativo el modelo comete {razon} falsos positivos.
La sección 2 de reports/problema_ml.md declaró que el falso positivo es el
error caro, así que esa razón es la cifra a vigilar al mover el umbral.
''')

**Cómo leer la matriz.** Las filas son lo que ocurrió; las columnas, lo que el
modelo dijo. La diagonal son los aciertos y las dos celdas restantes son los
dos errores, que **no cuestan lo mismo**:

- **Falso positivo** — el modelo da por superviviente a quien no lo fue. Para
  la aseguradora es una póliza mal tarifada: pérdida directa.
- **Falso negativo** — el modelo da por muerto a quien sobrevivió. Se cobra de
  más o se rechaza a un cliente viable: ingreso no percibido, no pérdida.

El `classification_report` descompone la exactitud por clase. Conviene mirar
el **recall de la clase "sobrevivió"**: es la proporción de supervivientes que
el modelo encuentra, y es donde la exactitud global engaña más.

### Paso 52. Curvas ROC y de precisión-recall

In [ ]:
# Los tres finalistas en el mismo eje, con sus probabilidades fuera de muestra
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

fig, (iz, de) = plt.subplots(1, 2, figsize=(13, 5))
for nombre in proba_oof.columns:
    RocCurveDisplay.from_predictions(y_train, proba_oof[nombre], name=nombre, ax=iz)
    PrecisionRecallDisplay.from_predictions(y_train, proba_oof[nombre], name=nombre, ax=de)

iz.plot([0, 1], [0, 1], "r--", lw=1, label="azar")
de.axhline(y_train.mean(), color="red", ls="--", lw=1,
           label=f"azar ({y_train.mean():.3f})")
iz.set_title("ROC — fuera de muestra")
de.set_title("Precisión-Recall — fuera de muestra")
iz.legend(loc="lower right")
de.legend(loc="lower left")
plt.tight_layout()
plt.show()

**¿Cuándo es más informativa la curva PR que la ROC?** Cuando la clase
positiva es minoritaria.

La ROC enfrenta el recall con la **tasa de falsos positivos**, cuyo
denominador son todos los negativos. Si los negativos abundan, añadir falsos
positivos mueve poco ese denominador y la curva se ve bien aunque la precisión
sea mala. La curva PR usa como denominador los **positivos predichos**, que
son pocos, y por eso castiga cada falso positivo.

Con 38 % de supervivientes el desbalance es moderado y las dos curvas cuentan
una historia parecida. Pero la línea de azar lo dice todo: en la ROC es la
diagonal, en la PR es una horizontal a la altura de la tasa base. **La curva
PR es la que corresponde al problema de la aseguradora**, porque su eje
vertical es exactamente la precisión que la sección 2 declaró prioritaria.

### Paso 53. Calibración y elección del umbral

`predict_proba` devuelve un puntaje, y `0.5` es una convención heredada, no una
decisión. Dos preguntas distintas:

1. **¿La probabilidad dice la verdad?** Un modelo está *calibrado* si, entre
   los pasajeros a los que asigna 0.7, sobrevivió cerca del 70 %.
2. **¿Dónde cortar?** Eso depende del costo de cada error, y se decide con
   datos de validación.

In [ ]:
# Calibración: lo que el modelo promete vs. lo que ocurrió
base = y_train.mean()
print(f"Brier de {LIDER}: {ml.brier(y_train, proba):.4f}    "
      f"(0 = perfecto; predecir siempre la tasa base da {base * (1 - base):.4f})")

cal = ml.calibration_table(y_train, proba, n_bins=8)
cal

In [ ]:
# El mismo contenido, en gráfico, con el n de cada tramo anotado
ok = cal[cal["n"] > 0]

plt.figure(figsize=(6.5, 6.5))
plt.plot([0, 1], [0, 1], "r--", lw=1, label="calibración perfecta")
plt.plot(ok["proba_media"], ok["tasa_real"], "o-", label=LIDER)
for _, f in ok.iterrows():
    plt.annotate(f"n={int(f['n'])}", (f["proba_media"], f["tasa_real"]),
                 textcoords="offset points", xytext=(7, -11), fontsize=8)
plt.xlabel("probabilidad que promete el modelo")
plt.ylabel("tasa real observada")
plt.title(f"Calibración de {LIDER} — fuera de muestra")
plt.legend()
plt.grid(alpha=.3)
plt.show()

**Cómo leer este gráfico.** Cada punto es un tramo de probabilidad. Si el
modelo estuviera calibrado, todos caerían sobre la diagonal.

- Punto **por debajo** de la diagonal: el modelo promete más de lo que ocurre
  (optimista).
- Punto **por encima**: promete menos (pesimista).

El `n` anotado es imprescindible: un tramo con 3 pasajeros puede desviarse 30
puntos por puro azar. Solo los tramos con decenas de pasajeros sostienen una
lectura. Por eso la tabla se calcula a mano y no con `calibration_curve`, que
descarta los tramos vacíos y no devuelve el tamaño de cada uno.

**El Brier** es el error cuadrático medio de la probabilidad. La vara no es
0.25 sino `p·(1−p)` con la tasa base: eso es lo que obtiene quien responde
siempre "38.4 %" sin mirar al pasajero.

In [ ]:
# Tres umbrales, tres políticas distintas
from sklearn.metrics import f1_score, precision_score, recall_score

f1_max = ml.best_threshold(y_train, proba, criterio="f1")
piso80 = ml.threshold_for_precision(y_train, proba, min_precision=0.80)


def fila(criterio, u):
    p = (proba >= u).astype(int)
    return {"criterio": criterio, "umbral": round(float(u), 4),
            "precision": round(precision_score(y_train, p, zero_division=0), 4),
            "recall": round(recall_score(y_train, p, zero_division=0), 4),
            "f1": round(f1_score(y_train, p, zero_division=0), 4),
            "falsos_positivos": int(((p == 1) & (y_train == 0)).sum()),
            "falsos_negativos": int(((p == 0) & (y_train == 1)).sum())}


print("¿se alcanzó el piso de precisión de 0.80?", piso80["alcanzado"])
pd.DataFrame([fila("convención (0.5)", 0.5),
              fila("máximo F1", f1_max["umbral"]),
              fila("precisión >= 0.80", piso80["umbral"])])

In [ ]:
# Qué cuesta mover el umbral: las tres métricas a lo largo de todo el rango
barrido = ml.threshold_sweep(y_train, proba)

plt.figure(figsize=(9, 5))
for col in ["precision", "recall", "f1"]:
    plt.plot(barrido["umbral"], barrido[col], label=col)
plt.axvline(0.5, color="gray", ls=":", lw=1.5, label="convención 0.5")
plt.axvline(f1_max["umbral"], color="tab:green", ls="--", lw=1,
            label=f"máx F1 ({f1_max['umbral']:.3f})")
plt.axvline(piso80["umbral"], color="tab:blue", ls="--", lw=1,
            label=f"precisión >= 0.80 ({piso80['umbral']:.3f})")
plt.xlabel("umbral de decisión")
plt.ylabel("valor")
plt.title(f"Qué cuesta mover el umbral — {LIDER}")
plt.legend()
plt.grid(alpha=.3)
plt.show()

**La decisión.** Las tres filas de la tabla son tres políticas, no tres
calidades de modelo: el modelo es el mismo y solo cambia dónde se corta.

- **Máximo F1** trata los dos errores como igual de caros. La sección 2
  declaró que no lo son, así que optimizar F1 contradice el criterio escrito.
- **Precisión ≥ 0.80** sube el umbral: menos falsos positivos (el error caro)
  a costa de muchos más falsos negativos. Es coherente con lo declarado, pero
  hay que mirar cuánto recall se sacrifica antes de aceptarlo.
- **0.5** no está justificado por nada salvo la costumbre.

El umbral elegido se registra aquí y se congela en la Parte 17 como
`THRESHOLD`. Anota la decisión y su razón en `reports/problema_ml.md`: elegir
el umbral después de ver el conjunto de prueba sería exactamente la trampa que
el protocolo de la sección 5 se comprometió a evitar.

### Paso 54. Análisis de errores por subgrupos

Una exactitud global promedia sobre grupos que el modelo trata de forma muy
distinta. Un modelo con 82 % de exactitud puede estar acertando el 95 % en un
grupo y el 55 % en otro.

In [ ]:
# Exactitud, precisión y recall dentro de cada subgrupo (peor primero)
con_tramo = X_train.copy()
tramo = pd.cut(con_tramo["Age"], bins=[0, 16, 30, 50, 100],
               labels=["<16", "16-30", "30-50", "50+"])
con_tramo["TramoEdad"] = tramo.astype(object).fillna("sin edad")

pd.concat([ml.subgroup_report(con_tramo, y_train, pred, by=v)
           for v in ["Sex", "Pclass", "TramoEdad"]], ignore_index=True)

In [ ]:
# Los 10 errores en los que el modelo estaba más seguro
COLS = ["Sex", "Pclass", "Age", "Fare", "Familia", "TTitulo", "TCabin",
        "real", "proba", "pred"]
ml.confident_errors(X_train, y_train, proba, n=10)[COLS]

**Qué buscar en estas dos tablas.**

En la primera, el `n` manda: un subgrupo con 20 pasajeros no sostiene una
conclusión por mala que se vea su exactitud. Lo que importa es un grupo
**grande** donde el modelo rinda mucho peor que en el resto.

En la segunda, un error con probabilidad 0.51 es ruido; uno con 0.95 significa
que el modelo tiene una creencia firme y equivocada. Si esos diez casos
comparten algo —el mismo título, la misma clase, familias grandes— eso señala
una **variable que falta**, no un modelo mal ajustado.

#### Reflexión ética

El modelo se apoya en `Sex` y `Pclass`, que son las variables que gobernaron
la evacuación de 1912. Predecir bien **lo que ocurrió** no implica que *deba*
ocurrir así, ni que esas variables sean admisibles para decidir quién recibe
un servicio hoy; en la mayoría de jurisdicciones, tarificar por sexo es
ilegal.

Hay además una asimetría en los errores por subgrupo que merece nombrarse: si
el modelo falla sistemáticamente más en un grupo, trasladarlo a producción
traslada también ese trato desigual. Anota en `reports/problema_ml.md` en qué
contextos usarías este modelo y en cuáles no.

### Paso 55. Curva de aprendizaje: ¿más datos o un modelo distinto?

La pregunta operativa es dónde invertir: en conseguir más datos o en cambiar
el modelo. La curva de aprendizaje la responde entrenando con fracciones
crecientes del conjunto y mirando cómo evolucionan las dos métricas.

In [ ]:
%%time
pipe_lider = ml.build_pipeline(ml.get_candidates()[LIDER])
curva = ml.learning_curve_table(pipe_lider, X_train, y_train)
curva

In [ ]:
# Las dos curvas con su banda de ±1 desviación entre pliegues
plt.figure(figsize=(8.5, 5))
for col, etiqueta, color in [("train", "entrenamiento", "tab:blue"),
                             ("val", "validación", "tab:orange")]:
    media, desv = curva[f"{col}_mean"], curva[f"{col}_std"]
    plt.plot(curva["n_entrenamiento"], media, "o-", color=color, label=etiqueta)
    plt.fill_between(curva["n_entrenamiento"], media - desv, media + desv,
                     alpha=.2, color=color)
plt.xlabel("pasajeros usados para entrenar")
plt.ylabel("ROC-AUC")
plt.title(f"Curva de aprendizaje — {LIDER}")
plt.legend()
plt.grid(alpha=.3)
plt.show()

**El diagnóstico tiene dos lecturas opuestas.**

| Lo que se ve | Nombre | Qué hacer |
|---|---|---|
| Las dos curvas **convergen en un valor bajo** | sesgo alto | El modelo es demasiado simple o faltan variables. Más datos no arreglan nada. |
| Queda una **brecha grande** que se estrecha al crecer `n` | varianza alta | Más datos o más regularización sí ayudan. |
| La curva de validación **se aplana** antes del final | saturación | Duplicar los datos daría muy poco. El margen está en el modelo o en las variables. |

Mira la columna `brecha` de la tabla: si apenas se mueve entre el primer y el
último tamaño, la curva ya saturó y el camino es el ajuste de hiperparámetros
(Parte 14) y la revisión de variables (Parte 15), no más pasajeros.

### Cierre de la Parte 13

Lo que queda registrado para las partes siguientes:

1. **El umbral elegido** y su justificación → `reports/problema_ml.md`, y en la
   Parte 17 como `THRESHOLD`.
2. **El desempeño por subgrupos** → entra en la ficha del modelo
   (`reports/model_card.md`, Paso 69).
3. **El diagnóstico de la curva de aprendizaje** → decide cuánto esperar del
   ajuste de hiperparámetros de la Parte 14.

El conjunto de prueba sigue sin abrirse.